# Expense Category Classification Notebook

This notebook builds and compares category prediction approaches for personal finance transactions.

In [1]:
!pip install pandas scikit-learn sentence-transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 51.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 46.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 55.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 17.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 41.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 53.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 20.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 30.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/

## 1. Environment Setup

Install required libraries for data processing and modeling.

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Data Loading and Inspection

Load the dataset and inspect its structure.

In [ ]:
url = "https://huggingface.co/buckets/Diamond-12324/Expense-Tracker/resolve/personal_expense_classification%20-%20Final.csv?download=true"

df = pd.read_csv(url, encoding="utf-8")

df.head()

,Date,Transaction Description,Category,Amount,Type,Year,Month,Day,Day_of_Week,Is_Weekend,Quarter
0,2020-01-01,Game purchase,Entertainment,237.87,Expense,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01,Gift received,Other Income,1791.21,Income,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01,Tuition fee,Education,241.05,Expense,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01,Insurance payment,Healthcare,130.88,Expense,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01,Bought clothes,Shopping,81.69,Expense,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df.shape)
print(df.columns)
df["Category"].value_counts()

(51482, 11)
Index(['Date', 'Transaction Description', 'Category', 'Amount', 'Type', 'Year',
       'Month', 'Day', 'Day_of_Week', 'Is_Weekend', 'Quarter'],
      dtype='object')


,count
Category,
Healthcare,3760
Shopping,3760
Transport,3730
Food,3726
Utilities,3561
Education,3504
Misc,3503
Housing,3499
Entertainment,3463


## 3. Feature Engineering

Clean text, extract date-based features, and encode target labels.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.strip()

df["text"] = df["Transaction Description"].apply(clean_text)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Is_Weekend"] = df["Day_of_Week"].isin([5, 6]).astype(int)
df["Quarter"] = df["Date"].dt.quarter

In [ ]:
le = LabelEncoder()
df["y"] = le.fit_transform(df["Category"])

In [ ]:
df["Type"] = df["Type"].map({
    "Expense": 0,
    "Income": 1
})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df, df["y"], test_size=0.2, random_state=42
)

## 4. Model 1: Keyword Rule-Based

Create simple keyword rules and evaluate baseline accuracy.

In [ ]:
keyword_rules = {
    "Entertainment": ["game", "movie", "netflix"],
    "Education": ["tuition", "course", "school"],
    "Healthcare": ["insurance", "hospital", "medicine"],
    "Shopping": ["clothes", "shopping", "buy"],
}

In [ ]:
def keyword_predict(text):
    for category, keywords in keyword_rules.items():
        for kw in keywords:
            if kw in text:
                return category
    return "Unknown"

In [ ]:
y_pred_kw = [keyword_predict(x) for x in X_test["text"]]

y_pred_kw_encoded = [
    le.transform([y])[0] if y in le.classes_ else -1
    for y in y_pred_kw
]

valid_idx = [i for i, y in enumerate(y_pred_kw_encoded) if y != -1]

acc_kw = accuracy_score(
    [y_test.iloc[i] for i in valid_idx],
    [y_pred_kw_encoded[i] for i in valid_idx]
)

print("Keyword Accuracy:", acc_kw)

Keyword Accuracy: 0.7871287128712872


## 5. Model 2: TF-IDF + Structured Features

Vectorize transaction text with TF-IDF, combine with numeric features, and train Logistic Regression.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_text = vectorizer.fit_transform(X_train["text"])
X_test_text = vectorizer.transform(X_test["text"])

In [ ]:
num_features = ["Amount", "Type", "Month", "Day_of_Week", "Is_Weekend"]

# Fill missing values
X_train_num = X_train[num_features].copy()
X_test_num = X_test[num_features].copy()

X_train_num = X_train_num.fillna(0)
X_test_num = X_test_num.fillna(0)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

In [ ]:
from scipy.sparse import hstack

X_train_combined = hstack([X_train_text, X_train_num])
X_test_combined = hstack([X_test_text, X_test_num])

In [ ]:
from sklearn.linear_model import LogisticRegression

tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_combined, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_tfidf = tfidf_model.predict(X_test_combined)

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("TF-IDF + Structured Accuracy:", acc_tfidf)

TF-IDF + Structured Accuracy: 0.9985432650286491


## 6. Model 3 (Optional): MiniLM + Structured Features

Optional section. Run this only if you want to compare against a transformer embedding model.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-MiniLM-L3-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
X_train_emb = embedder.encode(X_train["text"].tolist())
X_test_emb = embedder.encode(X_test["text"].tolist())

In [ ]:
X_train_final = np.hstack([X_train_emb, X_train_num])
X_test_final = np.hstack([X_test_emb, X_test_num])

In [ ]:
mini_model = LogisticRegression(max_iter=1000)
mini_model.fit(X_train_final, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_mini = mini_model.predict(X_test_final)

acc_mini = accuracy_score(y_test, y_pred_mini)

print("MiniLM + Structured Accuracy:", acc_mini)
print(classification_report(y_test, y_pred_mini, target_names=le.classes_))

MiniLM + Structured Accuracy: 0.9987374963581626
               precision    recall  f1-score   support

    Childcare       1.00      1.00      1.00       688
    Education       0.98      1.00      0.99       692
Entertainment       1.00      0.98      0.99       681
         Food       1.00      1.00      1.00       775
    Freelance       1.00      1.00      1.00       623
   Healthcare       1.00      1.00      1.00       735
      Housing       1.00      1.00      1.00       689
   Investment       1.00      1.00      1.00       631
         Misc       1.00      1.00      1.00       720
 Other Income       1.00      1.00      1.00       614
       Rental       1.00      1.00      1.00       604
       Salary       1.00      1.00      1.00       628
     Shopping       1.00      1.00      1.00       766
    Transport       1.00      1.00      1.00       741
    Utilities       1.00      1.00      1.00       710

     accuracy                           1.00     10297
    macro avg 

## 7. Model Comparison (Optional)

Compare all three model accuracies. If you skipped Model 3, ignore its result row.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Keyword-Based",
        "TF-IDF + Structured",
        "MiniLM + Structured"
    ],
    "Accuracy": [
        acc_kw,
        acc_tfidf,
        acc_mini
    ]
})

results

,Model,Accuracy
0,Keyword-Based,0.787129
1,TF-IDF + Structured,0.998543
2,MiniLM + Structured,0.998737


## 8. Quick User Input Test

Use this helper function to test a single transaction input with a selected model (`keyword`, `tfidf`, or optional `minilm`).

In [ ]:
def predict_transaction_category(
    description,
    amount=0.0,
    transaction_type="Expense",
    date_str="2026-01-01",
    model_name="tfidf",
):
    """Predict a category for one transaction using keyword, tfidf, or optional minilm."""
    model_name = model_name.lower().strip()

    cleaned = clean_text(description)
    parsed_date = pd.to_datetime(date_str)

    type_value = 1 if str(transaction_type).lower() == "income" else 0
    day_of_week = int(parsed_date.dayofweek)
    is_weekend = 1 if day_of_week in [5, 6] else 0

    input_num = pd.DataFrame(
        [{
            "Amount": float(amount),
            "Type": type_value,
            "Month": int(parsed_date.month),
            "Day_of_Week": day_of_week,
            "Is_Weekend": is_weekend,
        }]
    )
    input_num_scaled = scaler.transform(input_num)

    if model_name == "keyword":
        pred_label = keyword_predict(cleaned)
        return pred_label

    if model_name == "tfidf":
        input_text = vectorizer.transform([cleaned])
        input_features = hstack([input_text, input_num_scaled])
        pred_idx = tfidf_model.predict(input_features)[0]
        return le.inverse_transform([pred_idx])[0]

    if model_name == "minilm":
        if "embedder" not in globals() or "mini_model" not in globals():
            return "MiniLM model is optional and not loaded. Run the Model 3 section first."
        input_emb = embedder.encode([cleaned])
        input_features = np.hstack([input_emb, input_num_scaled])
        pred_idx = mini_model.predict(input_features)[0]
        return le.inverse_transform([pred_idx])[0]

    return "Invalid model_name. Use 'keyword', 'tfidf', or 'minilm'."


# Quick interactive test
user_desc = input("Enter transaction description: ")
user_amt = float(input("Enter amount (e.g., 15.5): "))
user_type = input("Enter type (Expense/Income): ")
user_date = input("Enter date (YYYY-MM-DD): ")
user_model = input("Choose model (keyword/tfidf/minilm): ")

print(
    "Predicted category:",
    predict_transaction_category(
        description=user_desc,
        amount=user_amt,
        transaction_type=user_type,
        date_str=user_date,
        model_name=user_model,
    ),
)